# Lecture 03: HDFS и Hive

Короткая практика: загрузим CSV в HDFS, посмотрим базовые команды `hdfs dfs`, а затем создадим Hive-таблицы через DBeaver.

Используем общие HDFS-пути:

- `/warehouse/events_raw` - сырые CSV-данные
- `/warehouse/events_partitioned` - партиционированная Hive-таблица


## 1. Подготовка путей

В ноутбуке работаем из контейнера Jupyter. Локальный файл доступен по пути `/materials/...`, а HDFS доступен через команду `hdfs dfs`.

In [1]:
LOCAL_EVENTS = "/materials/lecture_03_hadoop/data/events/events.csv"
HDFS_RAW_DIR = "/warehouse/events_raw"
HDFS_PARTITIONED_DIR = "/warehouse/events_partitioned"

LOCAL_EVENTS, HDFS_RAW_DIR, HDFS_PARTITIONED_DIR

('/materials/lecture_03_hadoop/data/events/events.csv',
 '/warehouse/events_raw',
 '/warehouse/events_partitioned')

## 2. Посмотреть HDFS

`hdfs dfs -ls` показывает файлы и директории в HDFS. Начнём с корня и общего warehouse.

In [2]:
!hdfs dfs -ls /

Found 3 items
drwxrwxrwx   - root  supergroup          0 2026-05-29 15:14 /tmp
drwxr-xr-x   - root  supergroup          0 2026-05-29 15:14 /user
drwxrwxrwt   - spark supergroup          0 2026-06-02 18:33 /warehouse


In [3]:
!hdfs dfs -ls /warehouse

Found 1 items
drwxr-xr-x   - spark supergroup          0 2026-06-02 18:33 /warehouse/raw


## 3. Очистить старый запуск

`-rm -r` удаляет директорию рекурсивно. `-skipTrash` удаляет сразу, без корзины. Ошибка `No such file or directory` безопасна, если папок ещё нет.

In [4]:
!hdfs dfs -rm -r -skipTrash {HDFS_RAW_DIR} {HDFS_PARTITIONED_DIR} 2>/dev/null || true

## 4. Создать папку и загрузить CSV

`-mkdir -p` создаёт директорию, включая родительские. `-put -f` загружает локальный файл в HDFS и перезаписывает его, если он уже есть.

In [5]:
!hdfs dfs -mkdir -p {HDFS_RAW_DIR}

In [6]:
!hdfs dfs -put -f {LOCAL_EVENTS} {HDFS_RAW_DIR}/

## 5. Проверить файл в HDFS

`-ls` проверяет наличие файла. `-cat` печатает содержимое. `-du -h` показывает размер в человекочитаемом формате.

In [7]:
!hdfs dfs -ls {HDFS_RAW_DIR}

Found 1 items
-rw-r--r--   1 spark supergroup        347 2026-06-04 11:34 /warehouse/events_raw/events.csv


In [8]:
!hdfs dfs -cat {HDFS_RAW_DIR}/events.csv

1,101,view,1001,web,2026-05-24 10:15:00
2,101,add_to_cart,1001,web,2026-05-24 10:17:00
3,102,view,1002,mobile,2026-05-24 11:05:00
4,103,view,1003,web,2026-05-25 09:20:00
5,101,purchase,1001,web,2026-05-25 12:30:00
6,104,view,1002,mobile,2026-05-25 13:10:00
7,104,add_to_cart,1002,mobile,2026-05-25 13:12:00
8,105,view,1004,web,2026-05-26 18:45:00


In [9]:
!hdfs dfs -du -h {HDFS_RAW_DIR}

347  347  /warehouse/events_raw/events.csv


## 6. Метаданные файла и блоки

HDFS хранит файл как набор блоков. Для маленького CSV обычно будет один блок, но команды ниже показывают те же принципы, что и для больших файлов.

В `hdfs dfs` удобно смотреть размер, replication factor, block size и checksum. Для детального просмотра блоков используется `hdfs fsck` на NameNode.

In [10]:
# -ls -h показывает размер файла, replication factor, владельца и путь.
!hdfs dfs -ls -h {HDFS_RAW_DIR}/events.csv

-rw-r--r--   1 spark supergroup        347 2026-06-04 11:34 /warehouse/events_raw/events.csv


In [11]:
# -stat выводит отдельные метаданные: имя, размер, replication, block size.
!hdfs dfs -stat 'name=%n size=%b replication=%r block_size=%o' {HDFS_RAW_DIR}/events.csv

name=events.csv size=347 replication=1 block_size=134217728


In [12]:
# -checksum нужен, чтобы проверить, что содержимое файла не изменилось.
!hdfs dfs -checksum {HDFS_RAW_DIR}/events.csv

/warehouse/events_raw/events.csv	MD5-of-0MD5-of-512CRC32C	000002000000000000000000f85642b4cfe0998382bc52d15e1f47ad


In [13]:
# -count показывает количество директорий, файлов и общий размер внутри пути.
!hdfs dfs -count -h {HDFS_RAW_DIR}

           1            1                347 /warehouse/events_raw


In [19]:
!hdfs fsck /warehouse/events_raw/events.csv -files -blocks -locations

fsck: Unknown command
Usage: hadoop fs [generic options]
	[-appendToFile <localsrc> ... <dst>]
	[-cat [-ignoreCrc] <src> ...]
	[-checksum [-v] <src> ...]
	[-chgrp [-R] GROUP PATH...]
	[-chmod [-R] <MODE[,MODE]... | OCTALMODE> PATH...]
	[-chown [-R] [OWNER][:[GROUP]] PATH...]
	[-concat <target path> <src path> <src path> ...]
	[-copyFromLocal [-f] [-p] [-l] [-d] [-t <thread count>] [-q <thread pool queue size>] <localsrc> ... <dst>]
	[-copyToLocal [-f] [-p] [-crc] [-ignoreCrc] [-t <thread count>] [-q <thread pool queue size>] <src> ... <localdst>]
	[-count [-q] [-h] [-v] [-t [<storage type>]] [-u] [-x] [-e] [-s] <path> ...]
	[-cp [-f] [-p | -p[topax]] [-d] [-t <thread count>] [-q <thread pool queue size>] <src> ... <dst>]
	[-createSnapshot <snapshotDir> [<snapshotName>]]
	[-deleteSnapshot <snapshotDir> <snapshotName>]
	[-df [-h] [<path> ...]]
	[-du [-s] [-h] [-v] [-x] <path> ...]
	[-expunge [-immediate] [-fs <path>]]
	[-find <path> ... <expression> ...]
	[-get [-f] [-p] [-crc] [-ignoreC

### Проверка блоков через fsck

Команда `fsck` показывает блоки файла и DataNode, где они лежат. Её удобнее запускать из терминала на хосте, потому что она выполняется в контейнере `namenode`:

```bash
docker exec namenode /opt/hadoop-3.2.1/bin/hdfs fsck /warehouse/events_raw/events.csv -files -blocks -locations
```

На что смотреть в выводе:

- `Total blocks` - сколько блоков у файла
- `Block size` - размер блока
- `replica(s)` - количество реплик
- `DatanodeInfo` / адрес DataNode - где физически лежит блок


## 7. Hive SQL через DBeaver

Теперь откройте DBeaver и подключитесь к HiveServer2:

- Driver: Apache Hive / Hive JDBC
- Host: `localhost`
- Port: `10000`
- JDBC URL: `jdbc:hive2://localhost:10000/default`

Откройте файл `materials/lecture_03_hadoop/hive_events.sql` и выполните его целиком.

SQL создаёт две внешние таблицы:

- `lecture_03.events_raw` читает CSV из `/warehouse/events_raw`
- `lecture_03.events_partitioned` хранит Parquet-данные в `/warehouse/events_partitioned` с партицией `event_date`


## 8. Что такое партиции

Партиция в Hive - это разбиение таблицы на HDFS-директории по значению колонки.

Для таблицы `events_partitioned` партиции будут выглядеть так:

```text
/warehouse/events_partitioned/event_date=2026-05-24
/warehouse/events_partitioned/event_date=2026-05-25
/warehouse/events_partitioned/event_date=2026-05-26
```

Если запрос содержит `WHERE event_date = '2026-05-24'`, Hive может читать только нужную папку, а не всю таблицу. Это ускоряет запросы по частым фильтрам, например по дате.

## 9. Проверить результат после SQL

После выполнения SQL в DBeaver проверьте, что Hive создал партиции в HDFS.

In [ ]:
!hdfs dfs -ls {HDFS_PARTITIONED_DIR}

In [ ]:
!hdfs dfs -ls {HDFS_PARTITIONED_DIR}/event_date=2026-05-24

In [ ]:
!hdfs dfs -du -h {HDFS_PARTITIONED_DIR}

## 10. Полезные команды HDFS

- `hdfs dfs -ls <path>` - список файлов и директорий
- `hdfs dfs -mkdir -p <path>` - создать директорию
- `hdfs dfs -put -f <local> <hdfs>` - загрузить файл в HDFS
- `hdfs dfs -cat <path>` - вывести файл в консоль
- `hdfs dfs -du -h <path>` - показать размер
- `hdfs dfs -stat <format> <path>` - вывести метаданные файла
- `hdfs dfs -checksum <path>` - посчитать checksum файла
- `hdfs dfs -count -h <path>` - посчитать файлы, директории и размер
- `hdfs fsck <path> -files -blocks -locations` - проверить блоки и расположение реплик
- `hdfs dfs -rm -r -skipTrash <path>` - удалить директорию рекурсивно
